
## PANORAMA GERAL DO SINAN


 Objetivo:
 Construir uma síntese epidemiológica integrada da dengue
 a partir dos documentos semânticos temáticos produzidos
 no Notebook 06.
#
 **Este notebook:**
 - não reprocessa os dados brutos do SINAN;
 - não refaz as análises epidemiológicas do Notebook 05;
 - não recria os documentos temáticos do Notebook 06;
 - integra evidências provenientes de diferentes domínios;
 - gera uma visão nacional transversal e determinística.

In [ ]:
# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

import json
from pathlib import Path

import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# ============================================================
# 2. CONFIGURAÇÃO GERAL
# ============================================================

ANO = 2026

PASTA_BASE = Path(
    "/content/drive/MyDrive/Doutorado/arbovirus_rag"
)

PASTA_DOCS = (
    PASTA_BASE
    / "data_docs"
    / "SINAN"
    / str(ANO)
)

PASTA_INVENTARIO = (
    PASTA_DOCS
    / "_inventario"
)

PASTA_PANORAMA_GERAL = (
    PASTA_DOCS
    / "panorama_geral"
)

PASTA_PANORAMA_GERAL.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Pasta de documentos:",
    PASTA_DOCS
)

print(
    "Pasta do panorama geral:",
    PASTA_PANORAMA_GERAL
)

In [ ]:
# ============================================================
# 2. VERIFICAR A ÁRVORE DATA_DOCS
# ============================================================

caminhos = [
    "/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs",
    "/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN",
    "/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026",
    "/content/drive/MyDrive/Doutorado/arbovirus_rag/data_docs/SINAN/2026/_inventario"
]

for caminho in caminhos:

    p = Path(caminho)

    print(
        f"{p.exists():5} | {caminho}"
    )

In [ ]:
# ============================================================
# 3. CARREGAR INVENTÁRIO DOS DOCUMENTOS SEMÂNTICOS
# ============================================================

caminho_inventario = (
    PASTA_INVENTARIO
    / f"inventario_documentos_semanticos_{ANO}.csv"
)

print(
    "Pasta inventário existe:",
    PASTA_INVENTARIO.exists()
)

print(
    "Arquivo inventário existe:",
    caminho_inventario.exists()
)

df_inventario = pd.read_csv(
    caminho_inventario
)

print(
    f"Documentos encontrados no inventário: "
    f"{len(df_inventario)}"
)

display(
    df_inventario.head()
)

In [ ]:
# ============================================================
# SELECIONAR PANORAMAS NACIONAIS
# ============================================================

TIPOS_PANORAMA_NACIONAL = [
    "panorama_clinico_nacional",
    "panorama_geografico_nacional",
    "panorama_temporal_nacional",
    "panorama_virologico_nacional",
    "panorama_desfechos_nacional",
    "panorama_obitos_nacional"
]

df_panorama_nacional = (
    df_inventario[
        df_inventario[
            "TIPO_DOCUMENTO"
        ].isin(
            TIPOS_PANORAMA_NACIONAL
        )
    ]
    .copy()
)

print(
    f"Panoramas nacionais encontrados: "
    f"{len(df_panorama_nacional)}"
)

display(
    df_panorama_nacional[
        [
            "DOCUMENT_ID",
            "DOMINIO",
            "TIPO_DOCUMENTO",
            "ARQUIVO_JSON"
        ]
    ]
)

In [ ]:
# ============================================================
# 5. CARREGAR DOCUMENTOS NACIONAIS
# ============================================================

documentos_nacionais = {}

for _, linha in df_panorama_nacional.iterrows():

    caminho_json = (
        PASTA_DOCS
        / linha["ARQUIVO_JSON"]
    )

    with open(
        caminho_json,
        "r",
        encoding="utf-8"
    ) as arquivo:

        documento = json.load(
            arquivo
        )

    tipo_documento = (
        documento["tipo_documento"]
    )

    documentos_nacionais[
        tipo_documento
    ] = documento


print(
    "Documentos nacionais carregados:",
    len(documentos_nacionais)
)

print()

for tipo in sorted(
    documentos_nacionais
):
    print("-", tipo)

In [ ]:
# ============================================================
# 6. INSPECIONAR INDICADORES DOS PANORAMAS NACIONAIS
# ============================================================

for tipo, documento in (
    documentos_nacionais.items()
):

    print("=" * 80)

    print(
        "TIPO:",
        tipo
    )

    print(
        "ID:",
        documento.get(
            "document_id"
        )
    )

    print(
        "DOMÍNIO:",
        documento.get(
            "dominio"
        )
    )

    print(
        "\nESCOPO:"
    )

    print(
        documento.get(
            "escopo",
            {}
        )
    )

    print(
        "\nINDICADORES:"
    )

    indicadores = documento.get(
        "indicadores",
        {}
    )

    for chave, valor in (
        indicadores.items()
    ):

        print(
            f"  {chave}: {valor}"
        )

    print()

In [ ]:
# ============================================================
# 7. INSPECIONAR TIPOS DOS INDICADORES
# ============================================================

for tipo, documento in (
    documentos_nacionais.items()
):

    print("=" * 80)
    print(tipo)

    indicadores = documento.get(
        "indicadores",
        {}
    )

    for chave, valor in indicadores.items():

        print(
            f"{chave:45} "
            f"{type(valor).__name__}"
        )

    print()

O que faremos em seguida

Com essa saída, vamos construir uma camada intermediária do Notebook 07:

In [ ]:
sintese_integrada = {
    "situacao_geral": {},
    "temporal": {},
    "geografico": {},
    "virologico": {},
    "clinico": {},
    "desfechos": {},
    "obitos": {}
}

O objetivo do Notebook 07 é sintetizar, não reproduzir o Notebook 06.

Por exemplo:
```
Panorama clínico
        ↓
sintoma predominante
doença preexistente predominante

Panorama virológico
        ↓
cobertura de sorotipagem
sorotipo predominante

Panorama temporal
        ↓
pico
tendência
janela temporal

Panorama geográfico
        ↓
concentração territorial
maior incidência
maior volume

Panorama desfechos
        ↓
hospitalização
evolução predominante

Panorama óbitos
        ↓
quantidade
categoria predominante
perfil dos registros
```

Isso mantém uma distinção metodológica importante:

- Notebook 06 = evidência semântica temática detalhada.

- Notebook 07 = síntese semântica epidemiológica transversal.

Há um ponto importante antes do código: o panorama virológico nacional não contém um sorotipo_predominante geral. Ele informa cobertura de sorotipagem — 26.324 registros com sorotipo informado (6,18%) e 399.605 sem informação (93,82%). Portanto, no Notebook 07 não devemos inventar um sorotipo predominante nacional a partir desse documento. Já no panorama de registros classificados em categorias de óbito, temos DENV-2 como predominante entre os 100 registros de óbito com sorotipo informado.

Também manteria total_registros como referência geral de registros notificados, e não “casos”, mesmo que alguns produtos geográficos ainda tenham campos históricos chamados total_casos_*.

## 7. Validar coerência do escopo temporal

Antes da síntese, vamos confirmar que os seis documentos se referem à mesma janela temporal

In [ ]:
# ============================================================
# 7. VALIDAR COERÊNCIA DO ESCOPO TEMPORAL
# ============================================================

escopos_temporais = []

for tipo, documento in documentos_nacionais.items():

    escopo = documento.get(
        "escopo",
        {}
    )

    indicadores = documento.get(
        "indicadores",
        {}
    )

    semana_inicial = escopo.get(
        "semana_inicial",
        indicadores.get("semana_inicial")
    )

    semana_final = escopo.get(
        "semana_final",
        indicadores.get("semana_final")
    )

    total_semanas = escopo.get(
        "total_semanas",
        indicadores.get("total_semanas")
    )

    escopos_temporais.append({
        "TIPO_DOCUMENTO": tipo,
        "ANO": escopo.get("ano", ANO),
        "SEMANA_INICIAL": semana_inicial,
        "SEMANA_FINAL": semana_final,
        "TOTAL_SEMANAS": total_semanas
    })


df_escopos_temporais = pd.DataFrame(
    escopos_temporais
)

display(
    df_escopos_temporais
)

In [ ]:
# ============================================================
# VERIFICAR CONSISTÊNCIA TEMPORAL
# ============================================================

colunas_escopo = [
    "ANO",
    "SEMANA_INICIAL",
    "SEMANA_FINAL",
    "TOTAL_SEMANAS"
]

for coluna in colunas_escopo:

    valores = (
        df_escopos_temporais[
            coluna
        ]
        .dropna()
        .unique()
    )

    print(
        f"{coluna}: {valores}"
    )

In [ ]:
# ============================================================
# 8. DEFINIR ESCOPO DO PANORAMA INTEGRADO
# ============================================================

SEMANA_INICIAL = int(
    df_escopos_temporais[
        "SEMANA_INICIAL"
    ].dropna().iloc[0]
)

SEMANA_FINAL = int(
    df_escopos_temporais[
        "SEMANA_FINAL"
    ].dropna().iloc[0]
)

TOTAL_SEMANAS = (
    SEMANA_FINAL
    - SEMANA_INICIAL
    + 1
)

ESCOPO_INTEGRADO = {
    "localizacao": "Brasil",
    "ano": ANO,
    "semana_inicial": SEMANA_INICIAL,
    "semana_final": SEMANA_FINAL,
    "total_semanas": TOTAL_SEMANAS,
    "populacao_analitica":
        "registros de dengue notificados no SINAN"
}

print(
    ESCOPO_INTEGRADO
)

## 9. Criar referências aos seis panoramas

Para evitar escrever repetidamente nomes longos:

In [ ]:
# ============================================================
# 9. REFERÊNCIAS AOS PANORAMAS NACIONAIS
# ============================================================

doc_clinico = documentos_nacionais[
    "panorama_clinico_nacional"
]

doc_geografico = documentos_nacionais[
    "panorama_geografico_nacional"
]

doc_temporal = documentos_nacionais[
    "panorama_temporal_nacional"
]

doc_virologico = documentos_nacionais[
    "panorama_virologico_nacional"
]

doc_desfechos = documentos_nacionais[
    "panorama_desfechos_nacional"
]

doc_obitos = documentos_nacionais[
    "panorama_obitos_nacional"
]

In [ ]:
# ============================================================
# INDICADORES POR DIMENSÃO
# ============================================================

ind_clinico = doc_clinico["indicadores"]
ind_geografico = doc_geografico["indicadores"]
ind_temporal = doc_temporal["indicadores"]
ind_virologico = doc_virologico["indicadores"]
ind_desfechos = doc_desfechos["indicadores"]
ind_obitos = doc_obitos["indicadores"]
# Autoctonia já integrada ao panorama geográfico nacional
ind_autoctonia = ind_geografico.get("autoctonia", {})
ind_consistencia_autoctonia = ind_geografico.get("consistencia_autoctonia", {})


## 10. Construir a síntese integrada estruturada

Agora começamos efetivamente o trabalho específico do Notebook 07.

In [ ]:
# ============================================================
# 10. CONSTRUIR SÍNTESE EPIDEMIOLÓGICA INTEGRADA
# ============================================================

sintese_integrada = {

    "situacao_geral": {

        "total_registros":
            ind_temporal[
                "total_registros"
            ],

        "semana_inicial":
            SEMANA_INICIAL,

        "semana_final":
            SEMANA_FINAL,

        "total_semanas":
            TOTAL_SEMANAS
    },

    "temporal": {

        "media_semanal":
            ind_temporal[
                "media_semanal"
            ],

        "mediana_semanal":
            ind_temporal[
                "mediana_semanal"
            ],

        "semana_pico":
            ind_temporal[
                "semana_pico"
            ],

        "total_registros_pico":
            ind_temporal[
                "total_pico"
            ],

        "percentual_pico_total":
            ind_temporal[
                "percentual_pico_total"
            ]
    },

    "geografico": {

        "total_ufs":
            ind_geografico[
                "total_ufs_residencia"
            ],

        "total_municipios_analisados":
            ind_geografico[
                "total_municipios_analisados"
            ],

        "uf_maior_numero_registros_residencia":
            ind_geografico[
                "uf_maior_numero_residencia"
            ],

        "total_registros_uf_maior_numero":
            ind_geografico[
                "total_maior_uf_residencia"
            ],

        "municipio_maior_numero_registros":
            ind_geografico[
                "municipio_maior_numero_casos"
            ],

        "total_registros_municipio_maior_numero":
            ind_geografico[
                "maior_numero_casos_municipio"
            ],

        "municipio_maior_incidencia":
            ind_geografico[
                "municipio_maior_incidencia"
            ],

        "maior_incidencia_100mil":
            ind_geografico[
                "maior_incidencia_100mil"
            ]
    },


    "autoctonia": {

        "total_registros":
            ind_autoctonia.get(
                "total_registros"
            ),

        "autoctones_residencia":
            ind_autoctonia.get(
                "autoctones_residencia"
            ),

        "percentual_autoctones":
            ind_autoctonia.get(
                "percentual_autoctones"
            ),

        "nao_autoctones_residencia":
            ind_autoctonia.get(
                "nao_autoctones_residencia"
            ),

        "percentual_nao_autoctones":
            ind_autoctonia.get(
                "percentual_nao_autoctones"
            ),

        "indeterminados":
            ind_autoctonia.get(
                "indeterminados"
            ),

        "percentual_indeterminados":
            ind_autoctonia.get(
                "percentual_indeterminados"
            ),

        "ausentes":
            ind_autoctonia.get(
                "ausentes"
            ),

        "percentual_ausentes":
            ind_autoctonia.get(
                "percentual_ausentes"
            ),

        "total_comparaveis_uf_residencia_infeccao":
            ind_autoctonia.get(
                "total_comparaveis_uf_residencia_infeccao"
            ),

        "percentual_mesma_uf_residencia_infeccao":
            ind_autoctonia.get(
                "percentual_mesma_uf_residencia_infeccao"
            ),

        "total_comparaveis_municipio_residencia_infeccao":
            ind_autoctonia.get(
                "total_comparaveis_municipio_residencia_infeccao"
            ),

        "percentual_mesmo_municipio_residencia_infeccao":
            ind_autoctonia.get(
                "percentual_mesmo_municipio_residencia_infeccao"
            ),

        "uf_provavel_infeccao_maior_frequencia":
            ind_geografico.get(
                "uf_provavel_infeccao_maior_frequencia"
            ),

        "total_uf_provavel_infeccao_maior_frequencia":
            ind_geografico.get(
                "total_uf_provavel_infeccao_maior_frequencia"
            ),

        "consistencia": {
            "consistente":
                ind_consistencia_autoctonia.get(
                    "consistente"
                ),
            "nao_avaliavel":
                ind_consistencia_autoctonia.get(
                    "nao_avaliavel"
                ),
            "inconsistente":
                ind_consistencia_autoctonia.get(
                    "inconsistente"
                )
        }
    },

    "virologico": {

        "sorotipo_informado":
            ind_virologico[
                "sorotipo_informado"
            ],

        "percentual_sorotipo_informado":
            ind_virologico[
                "percentual_sorotipo_informado"
            ],

        "sorotipo_nao_informado":
            ind_virologico[
                "sorotipo_nao_informado"
            ],

        "percentual_sorotipo_nao_informado":
            ind_virologico[
                "percentual_sorotipo_nao_informado"
            ],

        "ufs_sem_sorotipo_informado":
            ind_virologico[
                "ufs_sem_sorotipo_informado"
            ]
    },

    "clinico": {

        "sinal_clinico_maior_frequencia":
            ind_clinico[
                "sinal_clinico_maior_frequencia"
            ],

        "percentual_sinal_clinico":
            ind_clinico[
                "percentual_sinal_clinico"
            ],

        "doenca_preexistente_maior_frequencia":
            ind_clinico[
                "doenca_preexistente_maior_frequencia"
            ],

        "percentual_doenca_preexistente":
            ind_clinico[
                "percentual_doenca_preexistente"
            ]
    },

    "desfechos": {

        "hospitalizacao_avaliaveis":
            ind_desfechos[
                "hospitalizacao_avaliaveis"
            ],

        "hospitalizados":
            ind_desfechos[
                "hospitalizados"
            ],

        "percentual_hospitalizados":
            ind_desfechos[
                "percentual_hospitalizados"
            ],

        "total_evolucao_avaliavel":
            ind_desfechos[
                "total_evolucao_avaliavel"
            ],

        "evolucao_predominante":
            ind_desfechos[
                "evolucao_predominante"
            ],

        "percentual_evolucao_predominante":
            ind_desfechos[
                "percentual_evolucao_predominante"
            ]
    },

    "obitos": {

        "total_registros_categorias_obito":
            ind_obitos[
                "total_registros_obito"
            ],

        "tipo_obito_predominante":
            ind_obitos[
                "tipo_obito_predominante"
            ],

        "total_tipo_obito_predominante":
            ind_obitos[
                "total_tipo_obito_predominante"
            ],

        "percentual_tipo_obito_predominante":
            ind_obitos[
                "percentual_tipo_obito_predominante"
            ],

        "total_sorotipo_informado":
            ind_obitos[
                "total_sorotipo_informado"
            ],

        "sorotipo_predominante_entre_informados":
            ind_obitos[
                "sorotipo_predominante_informado"
            ],

        "percentual_sorotipo_predominante_entre_informados":
            ind_obitos[
                "percentual_sorotipo_predominante"
            ],

        "sinal_clinico_mais_frequente":
            ind_obitos[
                "sinal_clinico_mais_frequente"
            ],

        "percentual_sinal_clinico":
            ind_obitos[
                "percentual_sinal_clinico"
            ],

        "doenca_preexistente_mais_frequente":
            ind_obitos[
                "doenca_preexistente_mais_frequente"
            ],

        "percentual_doenca_preexistente":
            ind_obitos[
                "percentual_doenca_preexistente"
            ]
    }
}

In [ ]:
# ============================================================
# VISUALIZAR SÍNTESE INTEGRADA
# ============================================================

print(
    json.dumps(
        sintese_integrada,
        ensure_ascii=False,
        indent=2
    )
)

Estamos fazendo:
```
6 documentos semânticos nacionais
            ↓
seleção de indicadores representativos
            ↓
harmonização terminológica
            ↓
síntese_integrada
            ↓
panorama epidemiológico integrado
```

In [ ]:
# ============================================================
# 11. GERAR INTERPRETAÇÃO INTEGRADA
# ============================================================

def gerar_interpretacao_integrada(sintese):

    geral = sintese["situacao_geral"]
    temporal = sintese["temporal"]
    geografico = sintese["geografico"]
    autoctonia = sintese["autoctonia"]
    virologico = sintese["virologico"]
    clinico = sintese["clinico"]
    desfechos = sintese["desfechos"]
    obitos = sintese["obitos"]

    interpretacao = (
        f"No período entre as semanas epidemiológicas "
        f"{geral['semana_inicial']} e {geral['semana_final']} de {ANO}, "
        f"foram consideradas "
        f"{formatar_inteiro_br(geral['total_registros'])} notificações "
        f"de dengue no SINAN. "

        f"Foram registradas, em média, "
        f"{formatar_decimal_br(temporal['media_semanal'])} "
        f"notificações de dengue por semana. "
        f"O maior volume ocorreu na semana epidemiológica "
        f"{temporal['semana_pico']}, com "
        f"{formatar_inteiro_br(temporal['total_registros_pico'])} "
        f"notificações, correspondendo a "
        f"{formatar_percentual_br(temporal['percentual_pico_total'])} "
        f"do total do período. "

        f"Na distribuição geográfica, "
        f"{geografico['uf_maior_numero_registros_residencia']} apresentou "
        f"o maior número de notificações segundo residência, com "
        f"{formatar_inteiro_br(geografico['total_registros_uf_maior_numero'])} "
        f"notificações. "
        f"{geografico['municipio_maior_numero_registros']} apresentou "
        f"o maior volume municipal, com "
        f"{formatar_inteiro_br(geografico['total_registros_municipio_maior_numero'])} "
        f"notificações. "
        f"{geografico['municipio_maior_incidencia']} apresentou "
        f"a maior incidência observada, de "
        f"{formatar_decimal_br(geografico['maior_incidencia_100mil'])} "
        f"por 100 mil habitantes. "

        f"Quanto à classificação de autoctonia, "
        f"{formatar_percentual_br(autoctonia['percentual_autoctones'])} "
        f"dos registros foram classificados como autóctones, "
        f"{formatar_percentual_br(autoctonia['percentual_nao_autoctones'])} "
        f"como não autóctones, "
        f"{formatar_percentual_br(autoctonia['percentual_indeterminados'])} "
        f"como indeterminados e "
        f"{formatar_percentual_br(autoctonia['percentual_ausentes'])} "
        f"apresentaram informação ausente. "
        f"Entre os registros comparáveis, "
        f"{formatar_percentual_br(autoctonia['percentual_mesma_uf_residencia_infeccao'])} "
        f"apresentaram a mesma UF de residência e provável infecção, "
        f"e {formatar_percentual_br(autoctonia['percentual_mesmo_municipio_residencia_infeccao'])} "
        f"o mesmo município. "
        f"Entre as UFs identificadas como provável local de infecção, "
        f"{autoctonia['uf_provavel_infeccao_maior_frequencia']} apresentou "
        f"o maior número de registros, com "
        f"{formatar_inteiro_br(autoctonia['total_uf_provavel_infeccao_maior_frequencia'])} "
        f"notificações. "

        f"Quanto ao perfil virológico, o sorotipo foi informado em "
        f"{formatar_inteiro_br(virologico['sorotipo_informado'])} registros "
        f"({formatar_percentual_br(virologico['percentual_sorotipo_informado'])}), "
        f"enquanto "
        f"{formatar_inteiro_br(virologico['sorotipo_nao_informado'])} registros "
        f"({formatar_percentual_br(virologico['percentual_sorotipo_nao_informado'])}) "
        f"não apresentaram informação de sorotipo. "

        f"No perfil clínico, "
        f"{clinico['sinal_clinico_maior_frequencia']} foi o sinal clínico "
        f"mais frequente "
        f"({formatar_percentual_br(clinico['percentual_sinal_clinico'])}), "
        f"e {clinico['doenca_preexistente_maior_frequencia']} foi a doença "
        f"preexistente mais frequente "
        f"({formatar_percentual_br(clinico['percentual_doenca_preexistente'])}). "

        f"Entre os "
        f"{formatar_inteiro_br(desfechos['hospitalizacao_avaliaveis'])} "
        f"registros avaliáveis para hospitalização, "
        f"{formatar_inteiro_br(desfechos['hospitalizados'])} foram classificados "
        f"como hospitalizados "
        f"({formatar_percentual_br(desfechos['percentual_hospitalizados'])}). "

        f"Entre os "
        f"{formatar_inteiro_br(desfechos['total_evolucao_avaliavel'])} "
        f"registros com evolução avaliável, "
        f"{desfechos['evolucao_predominante']} foi a categoria predominante "
        f"({formatar_percentual_br(desfechos['percentual_evolucao_predominante'])}). "

        f"Foram identificados "
        f"{formatar_inteiro_br(obitos['total_registros_categorias_obito'])} "
        f"registros classificados em categorias de evolução relacionadas a óbito. "
        f"A categoria predominante foi "
        f"{obitos['tipo_obito_predominante']} "
        f"({formatar_percentual_br(obitos['percentual_tipo_obito_predominante'])}). "

        f"Nesse subconjunto, "
        f"{formatar_inteiro_br(obitos['total_sorotipo_informado'])} registros "
        f"apresentaram sorotipo informado. "
        f"Entre esses registros, "
        f"{obitos['sorotipo_predominante_entre_informados']} foi o sorotipo "
        f"mais frequente "
        f"({formatar_percentual_br(obitos['percentual_sorotipo_predominante_entre_informados'])})."
    )

    return interpretacao


Há uma vantagem metodológica importante nessa função: ela apenas verbaliza valores já calculados. Ela não conclui, por exemplo, que o pico temporal ocorreu por causa do DENV-2, nem que hipertensão causou determinado desfecho.

Depois disso, podemos criar o documento final.

In [ ]:
# ============================================================
# FUNÇÕES DE FORMATAÇÃO NUMÉRICA - PADRÃO BRASILEIRO
# ============================================================

def formatar_inteiro_br(valor):
    """
    Exemplo:
    425929 -> 425.929
    """
    if valor is None:
        return "-"

    return f"{int(valor):,}".replace(",", ".")


def formatar_decimal_br(valor, casas=2):
    """
    Exemplo:
    13066.65 -> 13.066,65
    """
    if valor is None:
        return "-"

    valor_formatado = (
        f"{float(valor):,.{casas}f}"
    )

    return (
        valor_formatado
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )


def formatar_percentual_br(valor, casas=2):
    """
    Exemplo:
    88.21 -> 88,21%
    """
    if valor is None:
        return "-"

    return (
        formatar_decimal_br(
            valor,
            casas
        )
        + "%"
    )

In [ ]:
print(
    formatar_inteiro_br(444266)
)

print(
    formatar_decimal_br(13066.65)
)

print(
    formatar_decimal_br(9084.82)
)

print(
    formatar_percentual_br(6.64)
)

In [ ]:
# ============================================================
# 12. GERAR TEXTO DA INTERPRETAÇÃO INTEGRADA
# ============================================================

interpretacao_integrada = (
    gerar_interpretacao_integrada(
        sintese_integrada
    )
)

print(
    interpretacao_integrada
)

In [ ]:
#Antes disso, confirme que a função também está definida:
print(
    "Função existe:",
    "gerar_interpretacao_integrada" in globals()
)

print(
    "Síntese integrada existe:",
    "sintese_integrada" in globals()
)

In [ ]:
print(
    "Interpretação integrada existe:",
    "interpretacao_integrada" in globals()
)

In [ ]:
# ============================================================
# 13. CRIAR DOCUMENTO SEMÂNTICO DO PANORAMA GERAL
# ============================================================

documento_panorama_geral = {

    "document_id":
        f"SINAN_DENGUE_{ANO}_PANORAMA_GERAL",

    "titulo":
        f"Panorama epidemiológico integrado da dengue no Brasil - {ANO}",

    "tipo_documento":
        "panorama_epidemiologico_integrado",

    "dominio":
        "epidemiologico",

    "fonte": {
        "sistema": "SINAN",
        "doenca": "dengue",
        "ano": ANO
    },

    "escopo":
        ESCOPO_INTEGRADO,

    "sintese": (
        f"Síntese epidemiológica integrada dos registros de dengue "
        f"notificados no SINAN no Brasil entre as semanas epidemiológicas "
        f"{SEMANA_INICIAL} e {SEMANA_FINAL} de {ANO}, reunindo informações "
        f"temporais, geográficas, de autoctonia e provável local de infecção, virológicas, clínicas e de desfechos."
    ),

    "indicadores":
        sintese_integrada,

    "evidencias": {
        "documentos_semanticos_origem": [
            documento["document_id"]
            for documento in documentos_nacionais.values()
        ]
    },

    "interpretacao":
        interpretacao_integrada,

    "observacoes_dados": [
        (
            "Os indicadores apresentados foram obtidos a partir dos "
            "documentos semânticos nacionais produzidos no Notebook 06."
        ),
        (
            "Os percentuais de hospitalização e evolução utilizam apenas "
            "os registros avaliáveis em cada variável."
        ),
        (
            "A análise de autoctonia depende do preenchimento do provável "
            "local de infecção; registros ausentes e indeterminados são "
            "mantidos explicitamente nos indicadores."
        ),
        (
            "Os percentuais de concordância entre residência e provável local "
            "de infecção foram calculados apenas entre os registros comparáveis, "
            "não correspondendo ao total de notificações do período."
        ),
        (
            "A informação de sorotipo apresenta elevada ausência de "
            "preenchimento, devendo ser interpretada considerando essa "
            "limitação."
        ),
        (
            "Os registros classificados em categorias relacionadas a "
            "óbito incluem diferentes categorias de evolução e não devem "
            "ser interpretados indistintamente como óbitos causados por dengue."
        ),
        (
            "As relações apresentadas neste documento são descritivas e "
            "não representam inferências causais."
        )
    ],

    "conceitos_semanticos": [
        "Dengue",
        "Registro epidemiológico",
        "Semana epidemiológica",
        "Distribuição temporal",
        "Distribuição geográfica",
        "Incidência",
        "Autoctonia",
        "Caso autóctone",
        "Provável local de infecção",
        "UF provável de infecção",
        "Município provável de infecção",
        "Sorotipo",
        "Sinal clínico",
        "Doença preexistente",
        "Hospitalização",
        "Evolução clínica",
        "Óbito"
    ],

    "palavras_chave": [
        "dengue",
        "SINAN",
        "panorama epidemiológico",
        "vigilância epidemiológica",
        "semana epidemiológica",
        "incidência",
        "autoctonia",
        "autóctone",
        "provável local de infecção",
        "UF provável de infecção",
        "município provável de infecção",
        "sorotipo",
        "sinais clínicos",
        "hospitalização",
        "evolução",
        "óbito"
    ]
}

In [ ]:
# ============================================================
# 14. VALIDAR ESTRUTURA DO DOCUMENTO PANORAMA GERAL
# ============================================================

CAMPOS_OBRIGATORIOS = [
    "document_id",
    "titulo",
    "tipo_documento",
    "dominio",
    "fonte",
    "escopo",
    "sintese",
    "indicadores",
    "evidencias",
    "interpretacao",
    "observacoes_dados",
    "conceitos_semanticos",
    "palavras_chave"
]

campos_ausentes = [
    campo
    for campo in CAMPOS_OBRIGATORIOS
    if campo not in documento_panorama_geral
]

print(
    "Campos ausentes:",
    campos_ausentes
)

print(
    "Document ID:",
    documento_panorama_geral.get("document_id")
)

print(
    "Tipo:",
    documento_panorama_geral.get("tipo_documento")
)

print(
    "Domínio:",
    documento_panorama_geral.get("dominio")
)

In [ ]:
# ============================================================
# 15. VALIDAR DOCUMENTOS DE ORIGEM
# ============================================================

documentos_origem = (
    documento_panorama_geral[
        "evidencias"
    ][
        "documentos_semanticos_origem"
    ]
)

print(
    "Quantidade de documentos de origem:",
    len(documentos_origem)
)

for document_id in documentos_origem:
    print("-", document_id)

In [ ]:
# ============================================================
# 16. CONVERTER PANORAMA GERAL PARA MARKDOWN
# ============================================================

def panorama_geral_para_markdown(documento):

    ind = documento["indicadores"]

    geral = ind["situacao_geral"]
    temporal = ind["temporal"]
    geografico = ind["geografico"]
    autoctonia = ind["autoctonia"]
    virologico = ind["virologico"]
    clinico = ind["clinico"]
    desfechos = ind["desfechos"]
    obitos = ind["obitos"]

    linhas = []

    # --------------------------------------------------------
    # TÍTULO
    # --------------------------------------------------------

    linhas.append(
        f"# {documento['titulo']}\n"
    )

    # --------------------------------------------------------
    # IDENTIFICAÇÃO
    # --------------------------------------------------------

    linhas.append("## Identificação\n")

    linhas.append(
        f"- **ID do documento:** "
        f"{documento['document_id']}"
    )

    linhas.append(
        f"- **Tipo de documento:** "
        f"{documento['tipo_documento']}"
    )

    linhas.append(
        f"- **Domínio:** "
        f"{documento['dominio']}"
    )

    linhas.append(
        f"- **Fonte:** "
        f"{documento['fonte']['sistema']}"
    )

    linhas.append(
        f"- **Doença:** "
        f"{documento['fonte']['doenca']}"
    )

    linhas.append("")

    # --------------------------------------------------------
    # ESCOPO
    # --------------------------------------------------------

    linhas.append("## Escopo\n")

    linhas.append(
        f"- **Localização:** "
        f"{documento['escopo']['localizacao']}"
    )

    linhas.append(
        f"- **Ano:** "
        f"{documento['escopo']['ano']}"
    )

    linhas.append(
        f"- **Semanas epidemiológicas:** "
        f"{documento['escopo']['semana_inicial']} "
        f"a {documento['escopo']['semana_final']}"
    )

    linhas.append(
        f"- **Total de semanas:** "
        f"{documento['escopo']['total_semanas']}"
    )

    linhas.append("")

    # --------------------------------------------------------
    # SÍNTESE
    # --------------------------------------------------------

    linhas.append("## Síntese epidemiológica\n")

    linhas.append(
        documento["sintese"]
    )

    linhas.append("")

    # --------------------------------------------------------
    # SITUAÇÃO GERAL
    # --------------------------------------------------------

    linhas.append("## Situação geral\n")

    linhas.append(
        f"- **Total de notificações:** "
        f"{formatar_inteiro_br(geral['total_registros'])}"
    )

    linhas.append(
        f"- **Período:** semanas epidemiológicas "
        f"{geral['semana_inicial']} a "
        f"{geral['semana_final']}"
    )

    linhas.append("")

    # --------------------------------------------------------
    # TEMPORAL
    # --------------------------------------------------------

    linhas.append("## Comportamento temporal\n")

    linhas.append(
        f"- **Média semanal de notificações:** "
        f"{formatar_decimal_br(temporal['media_semanal'])}"
    )

    linhas.append(
        f"- **Mediana semanal de notificações:** "
        f"{formatar_decimal_br(temporal['mediana_semanal'])}"
    )

    linhas.append(
        f"- **Semana de pico:** "
        f"{temporal['semana_pico']}"
    )

    linhas.append(
        f"- **Notificações na semana de pico:** "
        f"{formatar_inteiro_br(temporal['total_registros_pico'])}"
    )

    linhas.append(
        f"- **Percentual do total concentrado na semana de pico:** "
        f"{formatar_percentual_br(temporal['percentual_pico_total'])}"
    )

    linhas.append("")

    # --------------------------------------------------------
    # GEOGRÁFICO
    # --------------------------------------------------------

    linhas.append("## Distribuição geográfica\n")

    linhas.append(
        f"- **UF com maior número de notificações por residência:** "
        f"{geografico['uf_maior_numero_registros_residencia']}"
    )

    linhas.append(
        f"- **Total de notificações nessa UF:** "
        f"{formatar_inteiro_br(geografico['total_registros_uf_maior_numero'])}"
    )

    linhas.append(
        f"- **Município com maior número de notificações:** "
        f"{geografico['municipio_maior_numero_registros']}"
    )

    linhas.append(
        f"- **Total de notificações no município:** "
        f"{formatar_inteiro_br(geografico['total_registros_municipio_maior_numero'])}"
    )

    linhas.append(
        f"- **Município com maior incidência:** "
        f"{geografico['municipio_maior_incidencia']}"
    )

    linhas.append(
        f"- **Maior incidência observada:** "
        f"{formatar_decimal_br(geografico['maior_incidencia_100mil'])} "
        f"por 100 mil habitantes"
    )

    linhas.append("")


    # --------------------------------------------------------
    # AUTOCTONIA E PROVÁVEL LOCAL DE INFECÇÃO
    # --------------------------------------------------------

    linhas.append("## Autoctonia e provável local de infecção\n")

    linhas.append(
        f"- **Registros classificados como autóctones:** "
        f"{formatar_inteiro_br(autoctonia['autoctones_residencia'])} "
        f"({formatar_percentual_br(autoctonia['percentual_autoctones'])})"
    )

    linhas.append(
        f"- **Registros classificados como não autóctones:** "
        f"{formatar_inteiro_br(autoctonia['nao_autoctones_residencia'])} "
        f"({formatar_percentual_br(autoctonia['percentual_nao_autoctones'])})"
    )

    linhas.append(
        f"- **Registros com autoctonia indeterminada:** "
        f"{formatar_inteiro_br(autoctonia['indeterminados'])} "
        f"({formatar_percentual_br(autoctonia['percentual_indeterminados'])})"
    )

    linhas.append(
        f"- **Registros com informação de autoctonia ausente:** "
        f"{formatar_inteiro_br(autoctonia['ausentes'])} "
        f"({formatar_percentual_br(autoctonia['percentual_ausentes'])})"
    )

    linhas.append(
        f"- **Registros comparáveis entre UF de residência e provável infecção:** "
        f"{formatar_inteiro_br(autoctonia['total_comparaveis_uf_residencia_infeccao'])}"
    )

    linhas.append(
        f"- **Mesma UF de residência e provável infecção:** "
        f"{formatar_percentual_br(autoctonia['percentual_mesma_uf_residencia_infeccao'])}"
    )

    linhas.append(
        f"- **Registros comparáveis entre município de residência e provável infecção:** "
        f"{formatar_inteiro_br(autoctonia['total_comparaveis_municipio_residencia_infeccao'])}"
    )

    linhas.append(
        f"- **Mesmo município de residência e provável infecção:** "
        f"{formatar_percentual_br(autoctonia['percentual_mesmo_municipio_residencia_infeccao'])}"
    )

    linhas.append(
        f"- **UF provável de infecção mais frequente:** "
        f"{autoctonia['uf_provavel_infeccao_maior_frequencia']}"
    )

    linhas.append(
        f"- **Registros nessa UF provável de infecção:** "
        f"{formatar_inteiro_br(autoctonia['total_uf_provavel_infeccao_maior_frequencia'])}"
    )

    linhas.append("")

    # --------------------------------------------------------
    # VIROLÓGICO
    # --------------------------------------------------------

    linhas.append("## Perfil virológico\n")

    linhas.append(
        f"- **Registros com sorotipo informado:** "
        f"{formatar_inteiro_br(virologico['sorotipo_informado'])} "
        f"({formatar_percentual_br(virologico['percentual_sorotipo_informado'])})"
    )

    linhas.append(
        f"- **Registros sem sorotipo informado:** "
        f"{formatar_inteiro_br(virologico['sorotipo_nao_informado'])} "
        f"({formatar_percentual_br(virologico['percentual_sorotipo_nao_informado'])})"
    )

    linhas.append(
        f"- **UFs sem sorotipo informado:** "
        f"{virologico['ufs_sem_sorotipo_informado']}"
    )

    linhas.append("")

    # --------------------------------------------------------
    # CLÍNICO
    # --------------------------------------------------------

    linhas.append("## Perfil clínico\n")

    linhas.append(
        f"- **Sinal clínico mais frequente:** "
        f"{clinico['sinal_clinico_maior_frequencia']} "
        f"({formatar_percentual_br(clinico['percentual_sinal_clinico'])})"
    )

    linhas.append(
        f"- **Doença preexistente mais frequente:** "
        f"{clinico['doenca_preexistente_maior_frequencia']} "
        f"({formatar_percentual_br(clinico['percentual_doenca_preexistente'])})"
    )

    linhas.append("")

    # --------------------------------------------------------
    # DESFECHOS
    # --------------------------------------------------------

    linhas.append("## Hospitalização e evolução\n")

    linhas.append(
        f"- **Registros avaliáveis para hospitalização:** "
        f"{formatar_inteiro_br(desfechos['hospitalizacao_avaliaveis'])}"
    )

    linhas.append(
        f"- **Hospitalizados:** "
        f"{formatar_inteiro_br(desfechos['hospitalizados'])} "
        f"({formatar_percentual_br(desfechos['percentual_hospitalizados'])})"
    )

    linhas.append(
        f"- **Registros com evolução avaliável:** "
        f"{formatar_inteiro_br(desfechos['total_evolucao_avaliavel'])}"
    )

    linhas.append(
        f"- **Evolução predominante:** "
        f"{desfechos['evolucao_predominante']} "
        f"({formatar_percentual_br(desfechos['percentual_evolucao_predominante'])})"
    )

    linhas.append("")

    # --------------------------------------------------------
    # ÓBITOS
    # --------------------------------------------------------

    linhas.append(
        "## Perfil dos registros classificados em categorias de óbito\n"
    )

    linhas.append(
        f"- **Total de registros classificados em categorias de óbito:** "
        f"{formatar_inteiro_br(obitos['total_registros_categorias_obito'])}"
    )

    linhas.append(
        f"- **Categoria predominante:** "
        f"{obitos['tipo_obito_predominante']} "
        f"({formatar_percentual_br(obitos['percentual_tipo_obito_predominante'])})"
    )

    linhas.append(
        f"- **Registros com sorotipo informado nesse subconjunto:** "
        f"{formatar_inteiro_br(obitos['total_sorotipo_informado'])}"
    )

    linhas.append(
        f"- **Sorotipo predominante entre os registros "
        f"com sorotipo informado:** "
        f"{obitos['sorotipo_predominante_entre_informados']} "
        f"({formatar_percentual_br(obitos['percentual_sorotipo_predominante_entre_informados'])})"
    )

    linhas.append(
        f"- **Sinal clínico mais frequente:** "
        f"{obitos['sinal_clinico_mais_frequente']} "
        f"({formatar_percentual_br(obitos['percentual_sinal_clinico'])})"
    )

    linhas.append(
        f"- **Doença preexistente mais frequente:** "
        f"{obitos['doenca_preexistente_mais_frequente']} "
        f"({formatar_percentual_br(obitos['percentual_doenca_preexistente'])})"
    )

    linhas.append("")

    # --------------------------------------------------------
    # INTERPRETAÇÃO
    # --------------------------------------------------------

    linhas.append("## Interpretação dos resultados\n")

    linhas.append(
        documento["interpretacao"]
    )

    linhas.append("")

    # --------------------------------------------------------
    # OBSERVAÇÕES
    # --------------------------------------------------------

    linhas.append("## Observações sobre os dados\n")

    for observacao in documento[
        "observacoes_dados"
    ]:
        linhas.append(
            f"- {observacao}"
        )

    linhas.append("")

    # --------------------------------------------------------
    # CONCEITOS
    # --------------------------------------------------------

    linhas.append("## Conceitos semânticos\n")

    for conceito in documento[
        "conceitos_semanticos"
    ]:
        linhas.append(
            f"- {conceito}"
        )

    linhas.append("")

    # --------------------------------------------------------
    # PALAVRAS-CHAVE
    # --------------------------------------------------------

    linhas.append("## Palavras-chave\n")

    linhas.append(
        ", ".join(
            documento["palavras_chave"]
        )
    )

    return "\n".join(linhas)

In [ ]:
# ============================================================
# 17. GERAR E INSPECIONAR MARKDOWN
# ============================================================

markdown_panorama_geral = (
    panorama_geral_para_markdown(
        documento_panorama_geral
    )
)

print(
    markdown_panorama_geral
)

In [ ]:
# ============================================================
# 17.1 VALIDAR INCLUSÃO DA AUTOCTONIA NO PANORAMA GERAL
# ============================================================

print("=" * 70)
print("VALIDAÇÃO DA AUTOCTONIA NO PANORAMA GERAL")
print("=" * 70)

autoctonia_integrada = documento_panorama_geral[
    "indicadores"
].get(
    "autoctonia"
)

print(
    "Autoctonia presente nos indicadores:",
    autoctonia_integrada is not None
)

print(
    "Seção de autoctonia presente no Markdown:",
    "## Autoctonia e provável local de infecção"
    in markdown_panorama_geral
)

if autoctonia_integrada:

    print(
        "Autóctones:",
        formatar_inteiro_br(
            autoctonia_integrada[
                "autoctones_residencia"
            ]
        ),
        "("
        + formatar_percentual_br(
            autoctonia_integrada[
                "percentual_autoctones"
            ]
        )
        + ")"
    )

    print(
        "Ausentes:",
        formatar_inteiro_br(
            autoctonia_integrada[
                "ausentes"
            ]
        ),
        "("
        + formatar_percentual_br(
            autoctonia_integrada[
                "percentual_ausentes"
            ]
        )
        + ")"
    )

    print(
        "Mesma UF residência/provável infecção:",
        formatar_percentual_br(
            autoctonia_integrada[
                "percentual_mesma_uf_residencia_infeccao"
            ]
        )
    )

    print(
        "Mesmo município residência/provável infecção:",
        formatar_percentual_br(
            autoctonia_integrada[
                "percentual_mesmo_municipio_residencia_infeccao"
            ]
        )
    )


In [ ]:
# ============================================================
# 18. SALVAR PANORAMA GERAL
# ============================================================

import json

# ------------------------------------------------------------
# GARANTIR QUE A PASTA DE SAÍDA EXISTA
# ------------------------------------------------------------

PASTA_PANORAMA_GERAL.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# NOMES DOS ARQUIVOS
# ------------------------------------------------------------

arquivo_json = (
    PASTA_PANORAMA_GERAL
    / f"sinan_dengue_{ANO}_panorama_geral.json"
)

arquivo_markdown = (
    PASTA_PANORAMA_GERAL
    / f"sinan_dengue_{ANO}_panorama_geral.md"
)

# ------------------------------------------------------------
# SALVAR JSON
# ------------------------------------------------------------

with open(
    arquivo_json,
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        documento_panorama_geral,
        arquivo,
        ensure_ascii=False,
        indent=2
    )

# ------------------------------------------------------------
# SALVAR MARKDOWN
# ------------------------------------------------------------

with open(
    arquivo_markdown,
    "w",
    encoding="utf-8"
) as arquivo:

    arquivo.write(
        markdown_panorama_geral
    )

# ------------------------------------------------------------
# CONFIRMAÇÃO
# ------------------------------------------------------------

print("Arquivos salvos com sucesso:")
print()
print("JSON:")
print(arquivo_json)
print()
print("Markdown:")
print(arquivo_markdown)

In [ ]:
# ============================================================
# 19. VALIDAR ARQUIVOS SALVOS
# ============================================================

print("=" * 70)
print("VALIDAÇÃO DO PANORAMA GERAL")
print("=" * 70)

# ------------------------------------------------------------
# EXISTÊNCIA DOS ARQUIVOS
# ------------------------------------------------------------

print(
    "JSON existe:",
    arquivo_json.exists()
)

print(
    "Markdown existe:",
    arquivo_markdown.exists()
)

# ------------------------------------------------------------
# TAMANHO DOS ARQUIVOS
# ------------------------------------------------------------

if arquivo_json.exists():

    print(
        "Tamanho JSON:",
        arquivo_json.stat().st_size,
        "bytes"
    )

if arquivo_markdown.exists():

    print(
        "Tamanho Markdown:",
        arquivo_markdown.stat().st_size,
        "bytes"
    )

# ------------------------------------------------------------
# REABRIR JSON
# ------------------------------------------------------------

with open(
    arquivo_json,
    "r",
    encoding="utf-8"
) as arquivo:

    documento_validacao = json.load(
        arquivo
    )

# ------------------------------------------------------------
# VERIFICAÇÕES BÁSICAS
# ------------------------------------------------------------

print()
print(
    "Document ID:",
    documento_validacao.get(
        "document_id"
    )
)

print(
    "Tipo:",
    documento_validacao.get(
        "tipo_documento"
    )
)

print(
    "Domínio:",
    documento_validacao.get(
        "dominio"
    )
)

print(
    "Ano:",
    documento_validacao[
        "fonte"
    ].get(
        "ano"
    )
)

print(
    "Semana inicial:",
    documento_validacao[
        "escopo"
    ].get(
        "semana_inicial"
    )
)

print(
    "Semana final:",
    documento_validacao[
        "escopo"
    ].get(
        "semana_final"
    )
)

In [ ]:
# ============================================================
# 20. VALIDAR CONTEÚDO DO MARKDOWN
# ============================================================

with open(
    arquivo_markdown,
    "r",
    encoding="utf-8"
) as arquivo:

    markdown_validacao = arquivo.read()

verificacoes = {
    "Título": documento_panorama_geral["titulo"]
        in markdown_validacao,

    "Document ID": documento_panorama_geral["document_id"]
        in markdown_validacao,

    "Semana inicial": (
        str(
            documento_panorama_geral[
                "escopo"
            ][
                "semana_inicial"
            ]
        )
        in markdown_validacao
    ),

    "Semana final": (
        str(
            documento_panorama_geral[
                "escopo"
            ][
                "semana_final"
            ]
        )
        in markdown_validacao
    ),

    "Interpretação": (
        documento_panorama_geral[
            "interpretacao"
        ]
        in markdown_validacao
    )
}

for nome, resultado in verificacoes.items():

    status = (
        "OK"
        if resultado
        else "ERRO"
    )

    print(
        f"{nome}: {status}"
    )

In [ ]:
# ============================================================
# 22. ATUALIZAR INVENTÁRIO COM O PANORAMA GERAL
# ============================================================

arquivo_inventario = (
    PASTA_INVENTARIO
    / f"inventario_documentos_semanticos_{ANO}.csv"
)

# ------------------------------------------------------------
# CARREGAR INVENTÁRIO EXISTENTE
# ------------------------------------------------------------

df_inventario = pd.read_csv(
    arquivo_inventario
)

print(
    "Documentos no inventário antes:",
    len(df_inventario)
)

# ------------------------------------------------------------
# CRIAR REGISTRO DO PANORAMA GERAL
# ------------------------------------------------------------

registro_panorama = {
    "DOCUMENT_ID": documento_panorama_geral["document_id"],
    "DOMINIO": documento_panorama_geral["dominio"],
    "TIPO_DOCUMENTO": documento_panorama_geral["tipo_documento"],
    "TITULO": documento_panorama_geral["titulo"],
    "ANO": ANO,

    "ARQUIVO_JSON": str(
        arquivo_json.relative_to(
            PASTA_DOCS
        )
    ),

    "ARQUIVO_MARKDOWN": str(
        arquivo_markdown.relative_to(
            PASTA_DOCS
        )
    )
}

df_registro_panorama = pd.DataFrame(
    [registro_panorama]
)

display(
    df_registro_panorama
)

In [ ]:
# ============================================================
# 23. VERIFICAR SE O PANORAMA JÁ EXISTE NO INVENTÁRIO
# ============================================================

document_id_panorama = (
    documento_panorama_geral[
        "document_id"
    ]
)

existe_no_inventario = (
    df_inventario[
        "DOCUMENT_ID"
    ]
    .astype(str)
    .eq(document_id_panorama)
    .any()
)

print(
    "Panorama já existe no inventário:",
    existe_no_inventario
)

In [ ]:
# ============================================================
# 24. ADICIONAR PANORAMA AO INVENTÁRIO
# ============================================================

if not existe_no_inventario:

    df_inventario_atualizado = pd.concat(
        [
            df_inventario,
            df_registro_panorama
        ],
        ignore_index=True
    )

else:

    # Remove eventual versão anterior do panorama
    # e insere a versão atual.
    df_inventario_atualizado = (
        df_inventario[
            df_inventario[
                "DOCUMENT_ID"
            ].astype(str)
            != document_id_panorama
        ]
        .copy()
    )

    df_inventario_atualizado = pd.concat(
        [
            df_inventario_atualizado,
            df_registro_panorama
        ],
        ignore_index=True
    )

print(
    "Documentos no inventário atualizado:",
    len(df_inventario_atualizado)
)

In [ ]:
# ============================================================
# 25. SALVAR INVENTÁRIO ATUALIZADO
# ============================================================

df_inventario_atualizado.to_csv(
    arquivo_inventario,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Inventário atualizado salvo em:"
)

print(
    arquivo_inventario
)

In [ ]:
# ============================================================
# 26. VALIDAR INVENTÁRIO ATUALIZADO
# ============================================================

df_inventario_validacao = pd.read_csv(
    arquivo_inventario
)

print(
    "Total de documentos:",
    len(df_inventario_validacao)
)

print(
    "IDs duplicados:",
    df_inventario_validacao[
        "DOCUMENT_ID"
    ].duplicated().sum()
)

print()

display(
    df_inventario_validacao[
        df_inventario_validacao[
            "DOCUMENT_ID"
        ]
        == document_id_panorama
    ]
)